# Testing a Deployed Model Locally

Before a model goes to a cloud server it should pass a suite of local tests. Catching bugs locally is free; catching them after a cloud deployment costs money, time, and user trust. This notebook builds a minimal Flask API around a trained model and tests it for correctness, edge cases, and latency.

**Learning objectives**
1. Build a Flask `/predict` endpoint with input validation.
2. Write a happy-path test that checks output format and values.
3. Write edge-case tests for bad inputs and verify the API returns the right error code.
4. Measure p50 and p95 latency over 100 requests using `time.perf_counter()`.


## 1  Why test locally?

A cloud VM costs money the moment it starts. If your model crashes on an empty input, returns predictions in the wrong format, or takes 10 seconds per request, you want to know that before paying for cloud infrastructure. Local testing also gives a fast feedback loop: edit code, rerun, fix — no deploy cycle required.


In [1]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "flask", "scikit-learn", "joblib"],
               check=False)
print("Dependencies ready")


Dependencies ready


## 2  Train a model and build a Flask app

We train a RandomForest on Iris data, write it to disk, then define a Flask app with a `/predict` endpoint that validates input before calling the model.


In [2]:
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import joblib
import numpy as np

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42
)
clf = RandomForestClassifier(n_estimators=50, random_state=42)
clf.fit(X_train, y_train)

MODEL_PATH = "/tmp/test_iris_rf.joblib"
joblib.dump(clf, MODEL_PATH)
print(f"Model saved: test accuracy = {clf.score(X_test, y_test):.2%}")


<frozen importlib._bootstrap>:219: RuntimeWarning: scipy._lib.messagestream.MessageStream size changed, may indicate binary incompatibility. Expected 56 from C header, got 64 from PyObject


Model saved: test accuracy = 100.00%


In [3]:
%%writefile /tmp/flask_app.py
from flask import Flask, request, jsonify
import joblib
import numpy as np

app = Flask(__name__)
MODEL_PATH = "/tmp/test_iris_rf.joblib"
CLASS_NAMES = ["setosa", "versicolor", "virginica"]
clf = joblib.load(MODEL_PATH)

REQUIRED_KEYS = ["sepal_length", "sepal_width", "petal_length", "petal_width"]

@app.route("/health")
def health():
    return jsonify({"status": "ok"})

@app.route("/predict", methods=["POST"])
def predict():
    data = request.get_json(silent=True)

    # Validate: must be a JSON object
    if data is None or not isinstance(data, dict):
        return jsonify({"error": "Request body must be a JSON object"}), 400

    # Validate: all required keys present
    missing = [k for k in REQUIRED_KEYS if k not in data]
    if missing:
        return jsonify({"error": f"Missing fields: {missing}"}), 422

    # Validate: all values must be numeric
    try:
        features = [float(data[k]) for k in REQUIRED_KEYS]
    except (ValueError, TypeError):
        return jsonify({"error": "All feature values must be numeric"}), 422

    # Validate: no out-of-range values (iris features in cm, 0-15 is reasonable)
    if any(v < 0 or v > 15 for v in features):
        return jsonify({"error": "Feature values out of expected range [0, 15]"}), 422

    arr = np.array([features])
    proba = clf.predict_proba(arr)[0]
    class_idx = int(np.argmax(proba))
    return jsonify({
        "prediction": CLASS_NAMES[class_idx],
        "confidence": round(float(proba[class_idx]), 4)
    })

if __name__ == "__main__":
    app.run(port=5000, debug=False)


Overwriting /tmp/flask_app.py


## 3  Test 1 — Happy path

Flask's `test_client()` lets us call the app in-process without starting a real server. We send a valid Iris sample and check that the response has the expected shape.


In [4]:
import sys, json
sys.path.insert(0, "/tmp")

import importlib
import flask_app as fa
importlib.reload(fa)

client = fa.app.test_client()

# Known setosa sample
payload = {"sepal_length": 5.1, "sepal_width": 3.5, "petal_length": 1.4, "petal_width": 0.2}
resp = client.post("/predict", data=json.dumps(payload), content_type="application/json")
body = json.loads(resp.data)

print(f"Status code : {resp.status_code}")
print(f"Prediction  : {body['prediction']}")
print(f"Confidence  : {body['confidence']}")

assert resp.status_code == 200, f"Expected 200, got {resp.status_code}"
assert "prediction" in body
assert "confidence" in body
assert body["prediction"] in ["setosa", "versicolor", "virginica"]
assert 0.0 <= body["confidence"] <= 1.0
print("Happy-path test PASSED")


Status code : 200
Prediction  : setosa
Confidence  : 1.0
Happy-path test PASSED


## 4  Test 2 — Edge cases

Edge cases are inputs that a real user might accidentally (or intentionally) send. A well-built API must reject them with a 4xx status code, never a 500 crash.


In [5]:
# Case A: empty body
resp_a = client.post("/predict", data="", content_type="application/json")
print(f"Empty body      -> status {resp_a.status_code}: {json.loads(resp_a.data)}")
assert resp_a.status_code == 400

# Case B: missing required field
partial = {"sepal_length": 5.1, "sepal_width": 3.5, "petal_length": 1.4}
resp_b = client.post("/predict", data=json.dumps(partial), content_type="application/json")
print(f"Missing field   -> status {resp_b.status_code}: {json.loads(resp_b.data)}")
assert resp_b.status_code == 422

# Case C: string value instead of float
bad_type = {"sepal_length": "five", "sepal_width": 3.5, "petal_length": 1.4, "petal_width": 0.2}
resp_c = client.post("/predict", data=json.dumps(bad_type), content_type="application/json")
print(f"String value    -> status {resp_c.status_code}: {json.loads(resp_c.data)}")
assert resp_c.status_code == 422

# Case D: out-of-range value
oor = {"sepal_length": 999, "sepal_width": 3.5, "petal_length": 1.4, "petal_width": 0.2}
resp_d = client.post("/predict", data=json.dumps(oor), content_type="application/json")
print(f"Out-of-range    -> status {resp_d.status_code}: {json.loads(resp_d.data)}")
assert resp_d.status_code == 422

print("\nAll edge-case tests PASSED")


Empty body      -> status 400: {'error': 'Request body must be a JSON object'}
Missing field   -> status 422: {'error': "Missing fields: ['petal_width']"}
String value    -> status 422: {'error': 'All feature values must be numeric'}
Out-of-range    -> status 422: {'error': 'Feature values out of expected range [0, 15]'}

All edge-case tests PASSED


## 5  Test 3 — Latency

We send 100 requests, measure each with `time.perf_counter()`, and compute p50 and p95. The p95 percentile is more useful than the mean because it tells you the worst-case experience for the slowest 5% of requests — the ones that actually cause users to notice lag.


In [6]:
import time
import numpy as np

payload_str = json.dumps({"sepal_length": 5.1, "sepal_width": 3.5,
                           "petal_length": 1.4, "petal_width": 0.2})
latencies_ms = []

for _ in range(100):
    t0 = time.perf_counter()
    client.post("/predict", data=payload_str, content_type="application/json")
    latencies_ms.append((time.perf_counter() - t0) * 1000)

arr = np.array(latencies_ms)
print(f"100 requests completed")
print(f"  p50 (median)  : {np.percentile(arr, 50):.2f} ms")
print(f"  p95           : {np.percentile(arr, 95):.2f} ms")
print(f"  p99           : {np.percentile(arr, 99):.2f} ms")
print(f"  mean          : {arr.mean():.2f} ms")
print("\nNote: these are in-process numbers. Real HTTP adds ~1-5 ms network overhead.")


100 requests completed
  p50 (median)  : 1.37 ms
  p95           : 4.84 ms
  p99           : 7.24 ms
  mean          : 2.00 ms

Note: these are in-process numbers. Real HTTP adds ~1-5 ms network overhead.


## 6  Combined pass/fail report


In [7]:
# Re-run the happy path to get a fresh response
resp = client.post("/predict", data=payload_str, content_type="application/json")
body = json.loads(resp.data)

checks = {
    "happy_path_status_200"    : resp.status_code == 200,
    "has_prediction_key"       : "prediction" in body,
    "has_confidence_key"       : "confidence" in body,
    "confidence_in_0_1"        : 0.0 <= body.get("confidence", -1) <= 1.0,
    "empty_body_returns_400"   : resp_a.status_code == 400,
    "missing_field_returns_422": resp_b.status_code == 422,
    "wrong_type_returns_422"   : resp_c.status_code == 422,
    "out_of_range_returns_422" : resp_d.status_code == 422,
    "p95_under_500ms"          : float(np.percentile(arr, 95)) < 500,
}

for name, passed in checks.items():
    print(f"  [{'PASS' if passed else 'FAIL'}] {name}")

print(f"\n{'ALL TESTS PASSED' if all(checks.values()) else 'SOME TESTS FAILED'}")


  [PASS] happy_path_status_200
  [PASS] has_prediction_key
  [PASS] has_confidence_key
  [PASS] confidence_in_0_1
  [PASS] empty_body_returns_400
  [PASS] missing_field_returns_422
  [PASS] wrong_type_returns_422
  [PASS] out_of_range_returns_422
  [PASS] p95_under_500ms

ALL TESTS PASSED


## Summary

Testing a deployed model locally means running three categories of checks before touching a cloud environment:

| Test category | What it catches |
|---|---|
| Happy path | Basic correctness — model returns valid, properly shaped predictions |
| Edge cases | Bad inputs that should return 4xx, not crash with 500 |
| Latency | Whether the model meets performance requirements (check p95, not mean) |

Flask's `test_client()` lets you run all of this without starting a real server.


## Self-check

1. **What latency would you expect from a simple sklearn RandomForest, and how would that compare to a 7B parameter LLM?** Look at the numbers measured in the latency cell, then consider how much more computation a 7B-parameter autoregressive model requires.
2. **What HTTP status code should a validation error return — 400, 422, or 500?** Check what the Flask app returns for a missing field vs an empty body, and explain the semantic difference.
3. **Name two things that could pass all local tests but still fail in production.** Hint: think about differences between your laptop and a cloud server — OS libraries, Python versions, concurrent users, memory limits.
